In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
data_path = 'dblp-v10.csv'
df = pd.read_csv(data_path)

print('Original shape:', df.shape)
print('Original duplicates:', df.duplicated().sum())
print('Original missing values:\n', df.isna().sum().to_string())
display(df.head(3))

Original shape: (1000000, 8)
Original duplicates: 0
Original missing values:
 abstract      172467
authors            2
n_citation         0
references    124417
title              0
venue         177755
year               0
id                 0


,abstract,authors,n_citation,references,title,venue,year,id
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de


In [3]:
if 'df' not in globals() or 'venue' not in getattr(df, 'columns', []):
    df = pd.read_csv(data_path)

print('Original shape:', df.shape)
print('Original duplicates:', df.duplicated().sum())
print('Original missing values:\n', df.isna().sum().to_string())
display(df.head(3))

df.columns = df.columns.str.strip()
text_columns = ['abstract', 'authors', 'references', 'title', 'venue']

for column in text_columns:
    df[column] = df[column].fillna('').astype(str).str.strip()

n_citation = pd.to_numeric(df['n_citation'], errors='coerce')
year = pd.to_numeric(df['year'], errors='coerce')
df['n_citation'] = n_citation.fillna(n_citation.median()).astype(int)
df['year'] = year.fillna(year.median()).astype(int)

df = df.drop_duplicates().copy()
df = df[df['venue'] != ''].copy()

for column in ['abstract', 'authors', 'references', 'title']:
    df[column] = df[column].replace({'nan': '', 'None': ''}).fillna('').astype(str).str.strip()

print('Cleaned shape:', df.shape)
print('Cleaned duplicates:', df.duplicated().sum())
print('Remaining missing values:', int(df.isna().sum().sum()))
print('Blank venues:', int((df['venue'] == '').sum()))
display(df.head(3))

clean_df = df.copy()

Original shape: (1000000, 8)
Original duplicates: 0
Original missing values:
 abstract      172467
authors            2
n_citation         0
references    124417
title              0
venue         177755
year               0
id                 0


,abstract,authors,n_citation,references,title,venue,year,id
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de


Cleaned shape: (822245, 8)
Cleaned duplicates: 0
Remaining missing values: 0
Blank venues: 0


,abstract,authors,n_citation,references,title,venue,year,id
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de


In [4]:
df = clean_df.copy()

top_n = 10
rows_per_class = 3000

venue_counts = df['venue'].value_counts()
top_venues = venue_counts.head(top_n).index
df = df[df['venue'].isin(top_venues)].copy()

def sample_class(frame: pd.DataFrame) -> pd.DataFrame:
    sampled = frame.sample(n=min(len(frame), rows_per_class), random_state=42).copy()
    sampled['venue'] = frame.name
    return sampled

df = (
    df.groupby('venue', group_keys=False)
      .apply(sample_class)
      .reset_index(drop=True)
)


def build_feature_text(frame: pd.DataFrame) -> pd.Series:
    citation_bins = pd.cut(
        frame['n_citation'],
        bins=[-1, 0, 5, 20, 100, 500, 10**9],
        labels=['zero', 'low', 'medium', 'high', 'very_high', 'extreme']
    ).astype(str)
    return (
        frame['title'].astype(str) + ' ' +
        frame['abstract'].astype(str) + ' ' +
        frame['authors'].astype(str) + ' ' +
        frame['references'].astype(str) + ' ' +
        'year_' + frame['year'].astype(str) + ' ' +
        'citation_' + citation_bins
    ).str.replace(r'\s+', ' ', regex=True).str.strip()

df['feature_text'] = build_feature_text(df)
label_encoder = LabelEncoder()
df['venue_encoded'] = label_encoder.fit_transform(df['venue'])

print('Modeling shape:', df.shape)
print('Classes:', len(label_encoder.classes_))
print('Class distribution:\n', df['venue'].value_counts().head(10).to_string())
display(df[['venue', 'venue_encoded', 'feature_text']].head(3))

Modeling shape: (30000, 10)
Classes: 10
Class distribution:
 venue
Bioinformatics                                                          3000
Lecture Notes in Computer Science                                       3000
conference on decision and control                                      3000
global communications conference                                        3000
intelligent robots and systems                                          3000
international conference on acoustics, speech, and signal processing    3000
international conference on communications                              3000
international conference on image processing                            3000
international conference on robotics and automation                     3000
international geoscience and remote sensing symposium                   3000


,venue,venue_encoded,feature_text
0,Bioinformatics,0,Systemic properties of ensembles of metabolic ...
1,Bioinformatics,0,Identification of functional links between gen...
2,Bioinformatics,0,"SNPhood: investigate, quantify and visualise t..."


In [5]:
X = df['feature_text']
y = df['venue_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

Train size: 24000
Test size: 6000


In [6]:
candidate_models = {
    'MultinomialNB': MultinomialNB(),
    'ComplementNB': ComplementNB()
}

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [20000, 40000],
    'tfidf__min_df': [2],
    'nb__alpha': [0.1, 0.5, 1.0]
}

search_results = []
best_search = None
best_score = -1
best_name = None
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, estimator in candidate_models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('nb', estimator)
    ])
    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring='accuracy',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train)
    search_results.append({
        'model': name,
        'best_cv_accuracy': search.best_score_,
        'best_params': search.best_params_
    })
    if search.best_score_ > best_score:
        best_score = search.best_score_
        best_search = search
        best_name = name

results_df = pd.DataFrame(search_results).sort_values('best_cv_accuracy', ascending=False).reset_index(drop=True)
display(results_df)
print('Best model family:', best_name)
print('Best cross-validation accuracy:', best_score)
print('Best parameters:', best_search.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


Fitting 3 folds for each of 12 candidates, totalling 36 fits


,model,best_cv_accuracy,best_params
0,MultinomialNB,0.707125,"{'nb__alpha': 0.5, 'tfidf__max_features': 4000..."
1,ComplementNB,0.692875,"{'nb__alpha': 1.0, 'tfidf__max_features': 4000..."


Best model family: MultinomialNB
Best cross-validation accuracy: 0.707125
Best parameters: {'nb__alpha': 0.5, 'tfidf__max_features': 40000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}


In [7]:
best_model = best_search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print('Test accuracy:', test_accuracy)
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification report:\n', classification_report(y_test, y_pred, target_names=label_encoder.classes_))

sample_texts = X_test.iloc[:5] if hasattr(X_test, 'iloc') else X_test[:5]
sample_predictions = best_model.predict(sample_texts)
decoded_predictions = label_encoder.inverse_transform(sample_predictions)

print('Sample predictions:')
for text, predicted_label in zip(sample_texts, decoded_predictions):
    print('---')
    print('Predicted venue:', predicted_label)
    print('Text preview:', str(text)[:250])

Test accuracy: 0.7061666666666667
Confusion matrix:
 [[591   3   2   0   1   0   0   2   0   1]
 [ 11 430  30  22  18  19   6  52   7   5]
 [  3  12 516   8  26   9   6   2  17   1]
 [  0  23   5 356   0  13 198   3   0   2]
 [  0  10  21   2 365  12   0  21 166   3]
 [  4  14  17  28   2 387  27 112   1   8]
 [  0  31   3 271   4  28 252  11   0   0]
 [  1  17   2   5  13  45   4 494   4  15]
 [  1  21  30   1 221   4   1  25 291   5]
 [  0  11   1   1   1   7   0  24   0 555]]
Classification report:
                                                                       precision    recall  f1-score   support

                                                      Bioinformatics       0.97      0.98      0.98       600
                                   Lecture Notes in Computer Science       0.75      0.72      0.73       600
                                  conference on decision and control       0.82      0.86      0.84       600
                                    global communic

In [8]:
model_path = 'dblp_naive_bayes_model.joblib'
encoder_path = 'dblp_naive_bayes_label_encoder.joblib'

joblib.dump(best_model, model_path)
joblib.dump(label_encoder, encoder_path)


loaded_model = joblib.load(model_path)
loaded_encoder = joblib.load(encoder_path)

loaded_predictions = loaded_encoder.inverse_transform(loaded_model.predict(X_test.iloc[:3]))
print('Loaded model predictions:', list(loaded_predictions))

Loaded model predictions: ['international conference on robotics and automation', 'global communications conference', 'intelligent robots and systems']
